# Machine Learning Classification Evaluation

## Objective
The objective of this project is to evaluate and compare the performance of several machine learning algorithms on the breast cancer wisconsin dataset. 

The algorithms **SVM**, **KNN**, and **MLP** are implemented and assessed using **train-test-split** with appropriate evaluation metrics and visualizations. In addition, two **ensemble methods** are applied and their performance is compared with the base classifiers.

The project also includes:
* Dataset introduction
* Preprocessing steps
* Final comparative analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [6]:
df = pd.read_excel(r"D:\Other\Apply\CV\Projects\Machine Learning and Neural Networks (Theory and Practice)\Copy of breast-cancer-wisconsin.xlsx")
print(df.to_string(max_rows=8))      

          id  clump_thickness  size_uniformity  shape_uniformity  marginal_adhesion  epithelial_size bare_nucleoli  bland_chromatin  normal_nucleoli  mitoses  class
0    1048672                4                1                 1                  1                2             1                2                1        1      2
1    1049815                4                1                 1                  1                2             1                3                1        1      2
2    1050670               10                7                 7                  6                4            10                4                1        2      4
3    1050718                6                1                 1                  1                2             1                3                1        1      2
..       ...              ...              ...               ...                ...              ...           ...              ...              ...      ...    ...
679   8417

In [7]:
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 683 entries, 0 to 682
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 683 non-null    int64 
 1   clump_thickness    683 non-null    int64 
 2   size_uniformity    683 non-null    int64 
 3   shape_uniformity   683 non-null    int64 
 4   marginal_adhesion  683 non-null    int64 
 5   epithelial_size    683 non-null    int64 
 6   bare_nucleoli      683 non-null    object
 7   bland_chromatin    683 non-null    int64 
 8   normal_nucleoli    683 non-null    int64 
 9   mitoses            683 non-null    int64 
 10  class              683 non-null    int64 
dtypes: int64(10), object(1)
memory usage: 58.8+ KB


None

In [8]:
# drop the id column
df = df.drop(columns = ['id'])
# turns object data type numeric, removes rows with missing values, converts it to int.
df["bare_nucleoli"] = pd.to_numeric(df["bare_nucleoli"], errors="coerce")
df.dropna(axis=0, inplace = True) 
df["bare_nucleoli"] = df["bare_nucleoli"].astype(int)
print(df.to_string(max_rows=8))  
# removes duplicate rows
duplicates = df[df.duplicated()]
print("Duplicated Rows:\n", duplicates)
print("Number of duplicated rows:", df.duplicated().sum())
df.drop_duplicates(inplace=True)
display(df.info())

     clump_thickness  size_uniformity  shape_uniformity  marginal_adhesion  epithelial_size  bare_nucleoli  bland_chromatin  normal_nucleoli  mitoses  class
0                  4                1                 1                  1                2              1                2                1        1      2
1                  4                1                 1                  1                2              1                3                1        1      2
2                 10                7                 7                  6                4             10                4                1        2      4
3                  6                1                 1                  1                2              1                3                1        1      2
..               ...              ...               ...                ...              ...            ...              ...              ...      ...    ...
679                2                1                 1   

None

In [9]:
# separates features and target, label-encodes the target column.
x = df.drop(columns=["class"])
y = df["class"].map({2: 0, 4: 1})

In [ ]:
'''
# creates an iqr-based detector for non-outlier values of a column
def detect_outliers_iqr_mask(data, column):
  Q1 = data[column].quantile(0.25)
  Q3 = data[column].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  # Return a boolean Series indicating which rows are NOT outliers for this column
  return (data[column] >= lower_bound) & (data[column] <= upper_bound)
'''
'''
# builds a non-oulier-mask by iteratively performing logical ANDs between the row value (initially True) 
# and the individual masked column values of the data returned by the detector;
# the result determines whether a row should be dropped (False) or kept (True).

non_outlier_mask = pd.Series(True, index=X.index)
for column in X.columns:
    is_not_outlier_col = detect_outliers_iqr_mask(X, column)
    non_outlier_mask = non_outlier_mask & is_not_outlier_col
X_filtered = X[non_outlier_mask]
y_filtered = y[non_outlier_mask]
print(f"Original dataset shape (after NaN/duplicate removal): {X.shape}")
print(f"Dataset shape after removing outliers: {X_filtered.shape}")
'''
'''
# identifying and removing correlated features
corr_matrix = x.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
threshold = 0.85
to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
print(f"Features suggested for dropping (correlation > {threshold}): {to_drop}")
X_final = x.drop(columns=to_drop)
print(f"Shape after dropping correlated features: {X_final.shape}")
'''

In [ ]:
#  stratified k-fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Metrics to evaluate across all folds
scoring_metrics = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

# define base pipelines to avoid data leakage
svm_pipe = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42)),
    ]
)

knn_pipe = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5, metric="minkowski", p=2)),
    ]
)

mlp_pipe = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "mlp",
            MLPClassifier(
                hidden_layer_sizes=(16, 8),
                activation="relu",
                solver="adam",
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

# ensemble pipelines
estimators = [("svm", svm_pipe), ("knn", knn_pipe), ("mlp", mlp_pipe)]

voting_hard = VotingClassifier(estimators=estimators, voting="hard")
voting_soft = VotingClassifier(estimators=estimators, voting="soft")

# Put all models in a dictionary for streamlined execution
models = {
    "SVM (RBF)": svm_pipe,
    "KNN (k=5)": knn_pipe,
    "MLP (16, 8)": mlp_pipe,
    "Ensemble (Hard Voting)": voting_hard,
    "Ensemble (Soft Voting)": voting_soft,
}

Shape of scaled training data: (349, 9)
Shape of scaled testing data: (88, 9)


In [11]:
# classification using support vector machine
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.svm import SVC

# creates an svm capable of non-linear classification with regularization strength of 1.0, and automatic kernel coefficient
svm_classifier = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
# learning phase
svm_classifier.fit(X_train_scaled, y_train)
# predicts y_test values using x_test values
y_pred = svm_classifier.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9772727272727273
              precision    recall  f1-score   support

           0       1.00      0.95      0.97        41
           1       0.96      1.00      0.98        47

    accuracy                           0.98        88
   macro avg       0.98      0.98      0.98        88
weighted avg       0.98      0.98      0.98        88

[[39  2]
 [ 0 47]]


In [12]:
# classification using k-nearest neighbors
from sklearn.neighbors import KNeighborsClassifier

'''
calculates the distance of a new query from every single data point in the dataset using euclidean distance,
finds the 5 nearest data points,
takes a majority vote among those 5 neighbors; 
the class that appears most is the prediction for the label of the new query
'''
knn_classifier = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)

# learning and testing phase
knn_classifier.fit(X_train_scaled, y_train)
y_pred_knn = knn_classifier.predict(X_test_scaled)

# evaluation
print("--- KNN Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_knn):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))

--- KNN Performance ---
Accuracy: 0.9773

Confusion Matrix:
[[40  1]
 [ 1 46]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        41
           1       0.98      0.98      0.98        47

    accuracy                           0.98        88
   macro avg       0.98      0.98      0.98        88
weighted avg       0.98      0.98      0.98        88



In [13]:
# classification using multi-level perceptron
from sklearn.neural_network import MLPClassifier
mlp_model = MLPClassifier(hidden_layer_sizes=(16, 8), 
                          activation='relu', 
                          solver='adam', 
                          max_iter=1000, 
                          random_state=42)

mlp_model.fit(X_train_scaled, y_train)

y_pred_mlp = mlp_model.predict(X_test_scaled)

print("--- Standard MLP Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_mlp):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_mlp))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_mlp))

--- Standard MLP Performance ---
Accuracy: 0.9773

Confusion Matrix:
[[40  1]
 [ 1 46]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        41
           1       0.98      0.98      0.98        47

    accuracy                           0.98        88
   macro avg       0.98      0.98      0.98        88
weighted avg       0.98      0.98      0.98        88



In [14]:
# classification using multi-level perceptron with different activation functions
activations = ['identity', 'logistic', 'tanh', 'relu']
print("--- Comparing Activation Functions ---")
for af in activations:
    temp_mlp = MLPClassifier(hidden_layer_sizes=(16, 8), activation=af, solver='adam', max_iter=1000, random_state=42)
    temp_mlp.fit(X_train_scaled, y_train)
    y_pred = temp_mlp.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    print(f"Activation: {af:10} | Accuracy: {acc:.4f}")

--- Comparing Activation Functions ---
Activation: identity   | Accuracy: 0.9773
Activation: logistic   | Accuracy: 0.9886
Activation: tanh       | Accuracy: 0.9659
Activation: relu       | Accuracy: 0.9773


In [15]:
# classification using multi-level perceptron with different neuron counts
neuron_counts = [100, 50, 25, 10, 5]
print("--- Comparing Number of Neurons (Single Hidden Layer) ---")
for n in neuron_counts:
    temp_mlp = MLPClassifier(hidden_layer_sizes=(n,), activation='relu', solver='adam', max_iter=1000, random_state=42)
    temp_mlp.fit(X_train_scaled, y_train)
    y_pred = temp_mlp.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    print(f"Neurons: {n:3} | Accuracy: {acc:.4f}")

--- Comparing Number of Neurons (Single Hidden Layer) ---
Neurons: 100 | Accuracy: 0.9773
Neurons:  50 | Accuracy: 0.9886
Neurons:  25 | Accuracy: 0.9886
Neurons:  10 | Accuracy: 0.9773
Neurons:   5 | Accuracy: 0.9886


In [16]:
# classification using voting classifier
from sklearn.ensemble import VotingClassifier

# base models
clf_svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
clf_knn = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
clf_mlp = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=1000, random_state=42)

# hard & soft Voting
estimators = [('svm', clf_svm), ('knn', clf_knn), ('mlp', clf_mlp)]
voting_hard = VotingClassifier(estimators=estimators, voting='hard')
voting_soft = VotingClassifier(estimators=estimators, voting='soft')
voting_hard.fit(X_train_scaled, y_train)
voting_soft.fit(X_train_scaled, y_train)
y_pred_hard = voting_hard.predict(X_test_scaled)
y_pred_soft = voting_soft.predict(X_test_scaled)

In [17]:
# evaluation of hard voting
print("--- Ensemble (Hard Voting) Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_hard):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_hard))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_hard))

--- Ensemble (Hard Voting) Performance ---
Accuracy: 0.9886

Confusion Matrix:
[[40  1]
 [ 0 47]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99        41
           1       0.98      1.00      0.99        47

    accuracy                           0.99        88
   macro avg       0.99      0.99      0.99        88
weighted avg       0.99      0.99      0.99        88



In [18]:
# evaluation of soft voting
print("--- Ensemble (Soft Voting) Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_soft):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_soft))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_soft))

--- Ensemble (Soft Voting) Performance ---
Accuracy: 0.9886

Confusion Matrix:
[[40  1]
 [ 0 47]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99        41
           1       0.98      1.00      0.99        47

    accuracy                           0.99        88
   macro avg       0.99      0.99      0.99        88
weighted avg       0.99      0.99      0.99        88

